# 03 · Generate machine texts (5 LLM families via OpenRouter)
**Purpose:** for each validated human topic, generate **one paragraph** on the same topic with one generator (balanced round-robin), using the two COLING-2025 prompt templates (50/50). Human and machine texts are therefore **topic-paired**.

**v1.1:** planning is *stateful* — topics already generated are never re-assigned (no repeated cost); only new human topics are assigned, to the generators that are short of `per_cell × gen_margin` valid texts.

**Design guards:** request-level cache (resumable, never pays twice) · hard budget cap · actual serving provider and token usage logged per call · language-ID top-up pass so every (language × generator) cell reaches its target with *fresh* topics (no cherry-picking retries on the same topic).

**Outputs:** `01_raw/generations/{lang}.jsonl` (raw responses + metadata incl. per-call cost), `02_validated/generation_yield.csv`.
**Runtime:** CPU; ~1–3 h total (network-bound). **Cost estimate:** ≈ $10–40.

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "03_generate_machine_text"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); _ = env_snapshot(NOTEBOOK)

In [ ]:
!pip -q install fasttext

In [ ]:
import requests, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import userdata
OR_KEY = userdata.get("OPENROUTER_API_KEY")
migrate_config(); G = CFG["generation"]; D = CFG["data"]
GEN_IDS = list(GENERATORS.keys())
_lock = threading.Lock()

def spent_usd():
    tot = 0.0
    for lang in LANGS:
        for r in read_jsonl(P["raw_gen"] / f"{lang}.jsonl"):
            tot += float((r.get("usage") or {}).get("cost") or 0.0)
    return tot

def or_call(model, prompt, temperature, top_p, max_tokens, retries=6):
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": temperature,
            "top_p": top_p, "max_tokens": max_tokens, "usage": {"include": True}}
    for a in range(retries):
        try:
            r = requests.post("https://openrouter.ai/api/v1/chat/completions", timeout=G["timeout_s"],
                              headers={"Authorization": f"Bearer {OR_KEY}", "Content-Type": "application/json"}, json=body)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(min(60, 2 ** a)); continue
            r.raise_for_status(); return r.json()
        except requests.RequestException:
            time.sleep(min(60, 2 ** a))
    raise RuntimeError(f"OpenRouter failed after {retries} tries")

## 1 · Stateful assignment: new topics → generators that are below target

In [ ]:
per_cell = D["n_topics_target"] // len(GEN_IDS)               # 200 valid pairs per (language × generator)

def req_key(row):
    return stable_hash(json.dumps([row["model"], row["prompt"], G["temperature"], G["top_p"], G["max_tokens"]]), 16)

def gen_table(lang):
    """All cached generations of a language with validity flags (same normalisation and filters as notebook 04)."""
    g = pd.DataFrame(read_jsonl(P["raw_gen"] / f"{lang}.jsonl"))
    if g.empty:
        return g
    g = g[g.text.notna()].drop_duplicates("key")
    g["text_norm"] = g.text.map(lambda t: truncate_words(normalize_text(t), D["max_words"]))
    lid = langid(g.text_norm.tolist())
    g["lid_ok"] = [(l == LANG_META[lang]["lid"]) and (p >= D["langid_min_prob"]) for l, p in lid]
    g["len_ok"] = g.text_norm.map(n_words) >= D["min_words"]
    g["valid"] = g.lid_ok & g.len_ok
    return g

def plan_jobs(lang):
    hum = pd.read_parquet(P["validated"] / f"human_{lang}.parquet")
    hum = hum.assign(_h=hum.title.map(lambda t: stable_hash(t, 12))).sort_values("_h").reset_index(drop=True)
    g = gen_table(lang)
    used = set(g.title) if len(g) else set()                     # never re-generate a topic
    gv = g[g.title.isin(set(hum.title))] if len(g) else g        # count only generations whose human text still exists
    valid = gv[gv.valid].groupby("generator").size().reindex(GEN_IDS, fill_value=0) if len(gv) else pd.Series(0, index=GEN_IDS)
    fail = (1 - g.groupby("generator").valid.mean()).reindex(GEN_IDS).fillna(0.1) if len(g) else pd.Series(0.1, index=GEN_IDS)
    goal = int(np.ceil(per_cell * D["gen_margin"]))
    need = {gid: int(np.ceil(max(0, goal - valid[gid]) / max(0.5, 1 - fail[gid]))) for gid in GEN_IDS}
    n_p0 = (g[g.prompt_id == "p0"].groupby("generator").size().reindex(GEN_IDS, fill_value=0) if len(g) else pd.Series(0, index=GEN_IDS)).to_dict()
    n_all = (g.groupby("generator").size().reindex(GEN_IDS, fill_value=0) if len(g) else pd.Series(0, index=GEN_IDS)).to_dict()
    rank0 = int(g.topic_rank.max()) + 1 if len(g) else 0
    free = [h for h in hum.itertuples() if h.title not in used]
    rows, k = [], 0
    while any(v > 0 for v in need.values()) and k < len(free):
        for gid in GEN_IDS:
            if need[gid] <= 0 or k >= len(free):
                continue
            h = free[k]; k += 1; need[gid] -= 1
            pid = "p0" if n_p0[gid] <= n_all[gid] - n_p0[gid] else "p1"      # keep the two prompt templates 50/50
            n_all[gid] += 1; n_p0[gid] += pid == "p0"
            rows.append({"lang": lang, "title": h.title, "qid": h.qid, "pair_id": f"{lang}:{stable_hash(h.title)}",
                         "generator": gid, "model": GENERATORS[gid]["model"], "family": GENERATORS[gid]["family"],
                         "prompt_id": pid, "prompt": G["prompts"][pid].format(lang_name=LANG_META[lang]["name"], title=h.title),
                         "topic_rank": rank0 + len(rows)})
    if any(v > 0 for v in need.values()):
        log.warning(f"[{lang}] ran out of unused human topics; remaining need {need}")
    return pd.DataFrame(rows), valid

## 2 · Generate (cached, budget-capped)

In [ ]:
def run_generation(jobs: pd.DataFrame):
    by_lang = {}
    for lang, grp in jobs.groupby("lang"):
        cache = P["raw_gen"] / f"{lang}.jsonl"
        done = {r["key"] for r in read_jsonl(cache) if r.get("text")}
        todo = [r for r in grp.to_dict("records") if req_key(r) not in done]
        by_lang[lang] = (cache, todo)
    spent = spent_usd(); log.info(f"already spent ${spent:.2f} / cap ${G['budget_cap_usd']}")
    for lang, (cache, todo) in by_lang.items():
        log.info(f"[{lang}] {len(todo)} requests to send")
        def work(row):
            j = or_call(row["model"], row["prompt"], G["temperature"], G["top_p"], G["max_tokens"])
            ch = j["choices"][0]
            return row | {"key": req_key(row), "text": ch["message"].get("content"), "finish_reason": ch.get("finish_reason"),
                          "provider": j.get("provider"), "served_model": j.get("model"), "usage": j.get("usage"),
                          "temperature": G["temperature"], "top_p": G["top_p"], "time_utc": utcnow()}
        buf = []
        with ThreadPoolExecutor(G["n_workers"]) as ex:
            futs = [ex.submit(work, r) for r in todo]
            for i, f in enumerate(as_completed(futs), 1):
                try:
                    res = f.result(); buf.append(res); spent += float((res.get("usage") or {}).get("cost") or 0)
                except Exception as e:
                    log.warning(f"[{lang}] request failed: {e}")
                if len(buf) >= 50 or i == len(futs):
                    with _lock:
                        append_jsonl(cache, buf); buf = []
                    log.info(f"[{lang}] {i}/{len(futs)}  spent≈${spent:.2f}")
                if spent > G["budget_cap_usd"]:
                    with _lock:
                        append_jsonl(cache, buf); buf = []          # never lose paid-for results
                    log.error("BUDGET CAP REACHED — stopping"); ex.shutdown(cancel_futures=True); break
        if spent > G["budget_cap_usd"]:
            break

for round_ in range(3):                                         # first pass + up to 2 top-ups
    plans = {lang: plan_jobs(lang) for lang in LANGS}
    jobs = pd.concat([p[0] for p in plans.values()], ignore_index=True)
    log.info(f"round {round_ + 1}: valid per cell before = " + str({l: p[1].to_dict() for l, p in plans.items()}))
    if jobs.empty:
        break
    log.info(f"round {round_ + 1}: {len(jobs)} new requests\n{jobs.groupby(['lang', 'generator']).size().unstack(fill_value=0)}")
    run_generation(jobs)

## 3 · Yield table

In [ ]:
yt = []
for lang in LANGS:
    g = gen_table(lang)
    if len(g):
        yt.append(g.groupby("generator").agg(n=("valid", "size"), valid=("valid", "sum"),
                                             lid_fail=("lid_ok", lambda s: int((~s).sum())), short=("len_ok", lambda s: int((~s).sum())))
                  .reset_index().assign(lang=lang))
yt = pd.concat(yt, ignore_index=True)[["lang", "generator", "n", "valid", "lid_fail", "short"]]
yt["reached_goal"] = yt.valid >= int(np.ceil(per_cell * D["gen_margin"]))
yt.to_csv(P["validated"] / "generation_yield.csv", index=False); log.info(f"total spent ${spent_usd():.2f}")
yt.pivot(index="lang", columns="generator", values="valid")

**Report back:** the yield table and the total cost.